# Week 3: Data Cleaning & Outlier Detection - Sales Dataset
### Dataset: Pakistan E-Commerce Sales (89.984 baris x 23 kolom)

## Alur Cleaning yang Akan Dikerjakan:
```
STEP 1: Load Data & Investigasi Awal
  |
STEP 2: Deteksi Missing Value (Eksplisit & Implisit)
  |
STEP 3: Deteksi Inkonsistensi Teks (Typo, Spasi, Case)
  |
STEP 4: Standarisasi & Normalisasi Nilai Kategorikal
  |
STEP 5: Deteksi Invalid Data (Nilai Tidak Masuk Akal)
  |
STEP 6: Deteksi Outlier (IQR & Z-Score)
  |
STEP 7: Treatment & Export Dataset Bersih
```

---
## STEP 1: Load Data & Investigasi Awal
Tujuan: Memahami struktur dataset sebelum menyentuh apa pun.

In [18]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', 1000)

path = r'C:\Users\sayyi\Downloads\sales.csv'

# Definisikan teks yang harus dianggap NaN saat membaca CSV
missing_formats = ['None', 'none', 'null', '?', ' ', '   ', 'n/a', 'na', '--', '-']
df = pd.read_csv(path, na_values=missing_formats)

print('Ukuran dataset:', df.shape)
print('\nKolom & Tipe Data:')
print(df.dtypes)
print('\n5 Baris Pertama:')
df.head()

Ukuran dataset: (89984, 23)

Kolom & Tipe Data:
Order_ID                 str
Order_Date               str
Customer_ID              str
Customer_Age         float64
Gender                   str
City                     str
Region                   str
Customer_Segment         str
Sales_Channel            str
Product_ID               str
Product_Category         str
Product_Name             str
Unit_Price           float64
Quantity             float64
Discount_Percent     float64
Shipping_Cost        float64
Payment_Method           str
Order_Status             str
Return_Flag              str
Marketing_Channel        str
Customer_Rating      float64
Sales_Amount         float64
Profit_Margin        float64
dtype: object

5 Baris Pertama:


,Order_ID,Order_Date,Customer_ID,Customer_Age,Gender,City,Region,Customer_Segment,Sales_Channel,Product_ID,Product_Category,Product_Name,Unit_Price,Quantity,Discount_Percent,Shipping_Cost,Payment_Method,Order_Status,Return_Flag,Marketing_Channel,Customer_Rating,Sales_Amount,Profit_Margin
0,ORD-0000001,2024-01-09,CUS-020443,64.0,Female,Karachi,Sindh,Corporate,Retail Store,PRD-018,Office Supplies,Laptop Stand,3746.16,2.0,8.33,119.36,Credit Card,Delivered,No,Direct,5.0,6987.71,27.20
1,ORD-0000002,2025-06-13,CUS-025451,40.0,Male,Rawalpindi,Punjab,New,Marketplace,PRD-011,Home & Kitchen,Bedsheet Set,4203.05,1.0,9.38,167.96,Cash on Delivery,Delivered,No,Social Media,5.0,3976.91,31.46
2,ORD-0000003,2025-09-25,CUS-022569,56.0,Male,Lahore,Punjab,Returning,Marketplace,PRD-021,Toys & Games,Board Game,2723.85,5.0,17.63,250.01,Debit Card,Delivered,No,Search,3.0,11467.71,30.81
3,ORD-0000004,2024-01-01,CUS-010322,31.0,male,Rawalpindi,Punjab,Loyal,Marketplace,PRD-017,Office Supplies,Office Chair,24977.53,3.0,8.10,NaN,Cash on Delivery,Delivered,No,Direct,5.0,69182.12,25.20
4,ORD-0000005,2024-01-29,CUS-008308,42.0,Female,Gujranwala,Punjab,Loyal,Marketplace,PRD-012,Home & Kitchen,Desk Lamp,2995.38,4.0,15.37,348.78,Cash on Delivery,Delivered,No,Social Media,NaN,10489.07,26.54


---
## STEP 2: Deteksi Missing Value
Ada 2 jenis missing:
- **Eksplisit (NaN):** Cell benar-benar kosong, terdeteksi oleh `isnull()`
- **Implisit:** Cell terisi tapi nilainya tidak valid (misal: Age=0, Price negatif)

In [19]:
# 2A. Missing Value Eksplisit (NaN)
null_counts = df.isnull().sum()
null_percent = (df.isnull().mean() * 100).round(2)

summary = pd.DataFrame({
    'Missing': null_counts,
    'Persen (%)': null_percent
}).sort_values('Missing', ascending=False)

# Tampilkan hanya kolom yang punya missing
print('=== MISSING VALUE EKSPLISIT (NaN) ===')
display(summary[summary['Missing'] > 0])

=== MISSING VALUE EKSPLISIT (NaN) ===


,Missing,Persen (%)
Customer_Rating,22558,25.07
Marketing_Channel,7619,8.47
Shipping_Cost,6779,7.53
Payment_Method,5959,6.62
Customer_Age,4966,5.52
Customer_Segment,4121,4.58
Discount_Percent,4058,4.51
Profit_Margin,3224,3.58
Gender,3161,3.51
Unit_Price,2516,2.80


In [20]:
# 2B. Missing Value Implisit (Angka yang Tidak Masuk Akal)
print('=== DATA IMPLISIT BERMASALAH ===')
print(f"Age = 0 (bayi baru lahir beli online?)   : {(df['Customer_Age'] == 0).sum()}")
print(f"Age > 120 (tidak realistis)               : {(df['Customer_Age'] > 120).sum()}")
print(f"Unit Price < 0 (harga negatif?)           : {(df['Unit_Price'] < 0).sum()}")
print(f"Quantity <= 0 (beli 0 atau negatif item?) : {(df['Quantity'] <= 0).sum()}")
print(f"Discount < 0 (diskon negatif?)            : {(df['Discount_Percent'] < 0).sum()}")
print(f"Discount > 100 (diskon > 100%?)           : {(df['Discount_Percent'] > 100).sum()}")
print(f"Profit Margin < 0 (rugi)                  : {(df['Profit_Margin'] < 0).sum()}")

=== DATA IMPLISIT BERMASALAH ===
Age = 0 (bayi baru lahir beli online?)   : 9
Age > 120 (tidak realistis)               : 8
Unit Price < 0 (harga negatif?)           : 17
Quantity <= 0 (beli 0 atau negatif item?) : 35
Discount < 0 (diskon negatif?)            : 7
Discount > 100 (diskon > 100%?)           : 8
Profit Margin < 0 (rugi)                  : 285


In [21]:
# 2C. Lihat Baris-Baris yang Bermasalah (BIAR LU LIHAT SENDIRI DATANYA)

# Tampilkan baris dengan Age > 120
print('=== BARIS DENGAN AGE > 120 ===')
display(df.loc[df['Customer_Age'] > 120, ['Order_ID', 'Customer_Age', 'City', 'Sales_Amount']])

# Tampilkan baris dengan harga negatif
print('\n=== BARIS DENGAN UNIT PRICE NEGATIF ===')
display(df.loc[df['Unit_Price'] < 0, ['Order_ID', 'Product_Name', 'Unit_Price', 'Quantity', 'Sales_Amount']])

# Tampilkan baris dengan quantity negatif atau 0
print('\n=== BARIS DENGAN QUANTITY <= 0 ===')
display(df.loc[df['Quantity'] <= 0, ['Order_ID', 'Product_Name', 'Unit_Price', 'Quantity', 'Sales_Amount']])

=== BARIS DENGAN AGE > 120 ===


,Order_ID,Customer_Age,City,Sales_Amount
556,ORD-0000557,132.0,Lahore,1429.97
15300,ORD-0015301,132.0,Islamabad,11384.45
27423,ORD-0027424,132.0,Islamabad,14873.76
41045,ORD-0041046,132.0,Gujranwala,6104.01
66282,ORD-0066283,132.0,Lahore,3458.93
72421,ORD-0072422,132.0,Lahore,25419.43
81461,ORD-0081462,132.0,Sialkot,13905.92
82467,ORD-0082468,132.0,Lahore,3411.95



=== BARIS DENGAN UNIT PRICE NEGATIF ===


,Order_ID,Product_Name,Unit_Price,Quantity,Sales_Amount
806,ORD-0000807,Wireless Headphones,-250.0,1.0,7479.24
7561,ORD-0007562,Denim Jeans,-1.0,1.0,3447.81
15907,ORD-0015908,Face Serum,-250.0,1.0,2500.42
19072,ORD-0019073,NaN,-1.0,3.0,10085.33
19553,ORD-0019554,Smart Watch Strap,-250.0,3.0,4004.38
36474,ORD-0036475,Wireless Headphones,-250.0,2.0,18181.68
42384,ORD-0042385,Travel Backpack,-1.0,2.0,6929.71
42432,ORD-0042433,Cotton T-Shirt,-250.0,2.0,2720.55
43211,ORD-0043212,Air Fryer,-250.0,3.0,45899.76
49273,ORD-0049274,Face Serum,-1.0,2.0,4206.39



=== BARIS DENGAN QUANTITY <= 0 ===


,Order_ID,Product_Name,Unit_Price,Quantity,Sales_Amount
1186,ORD-0001187,Desk Lamp,2781.56,-1.0,2571.23
4306,ORD-0004307,Skin Care Kit,4894.34,-1.0,6347.68
5743,ORD-0005744,Power Bank,3521.59,0.0,3127.87
10399,ORD-0010400,Wireless Headphones,7540.41,-1.0,22797.66
10666,ORD-0010667,Building Blocks,2974.80,0.0,3119.71
16589,ORD-0016590,Laptop Stand,3571.48,-1.0,6847.84
17537,ORD-0017538,Running Shoes,5593.12,0.0,10102.59
21627,ORD-0021628,Running Shoes,NaN,0.0,10274.39
21631,ORD-0021632,Notebook Pack,775.87,0.0,3825.10
22089,ORD-0022090,NaN,1537.78,-3.0,5374.09


---
## STEP 3: Deteksi Inkonsistensi Teks
Masalah klasik: data yang sama ditulis berbeda-beda.

Contoh di dataset ini:
- Gender: `Male`, `male`, `MALE`, ` MALE `, `M` (5 versi untuk hal yang sama!)
- Payment: `Cash on Delivery`, `COD`, `cash on delivery`, ` CASH ON DELIVERY ` (4 versi!)

Kalau tidak distandardisasi, `groupby()` dan `value_counts()` akan menghitung mereka terpisah.

In [22]:
# Lihat semua variasi penulisan di kolom-kolom kategorikal
cat_cols = ['Gender', 'Customer_Segment', 'Sales_Channel', 'Payment_Method', 'Marketing_Channel']

for col in cat_cols:
    print(f'\n=== {col} ({df[col].nunique()} variasi unik) ===')
    print(df[col].value_counts(dropna=False))
    print()


=== Gender (10 variasi unik) ===
Gender
Male                 40286
Female               39375
NaN                   3161
Prefer not to say     2983
Non-binary            1318
 FEMALE                517
male                   488
 MALE                  487
F                      473
M                      452
female                 444
Name: count, dtype: int64


=== Customer_Segment (8 variasi unik) ===
Customer_Segment
Returning      36637
New            21434
Loyal          19740
Corporate       5936
NaN             4121
returning        712
 RETURNING       648
loyal            395
 LOYAL           361
Name: count, dtype: int64


=== Sales_Channel (10 variasi unik) ===
Sales_Channel
Online            39279
Marketplace       21168
Retail Store      20455
Phone Order        5289
NaN                1675
 ONLINE             502
E-commerce          476
online              470
retail store        242
Retail              232
 RETAIL STORE       196
Name: count, dtype: int64


=== Payment_

---
## STEP 4: Standarisasi Nilai Kategorikal
Treatment: `.str.strip()` (hapus spasi), `.str.lower()` (seragamkan huruf), lalu `.replace()` / `.map()` untuk mapping alias.

In [23]:
# 4A. GENDER: Standarisasi dari 10 variasi menjadi 4 kategori bersih
# Langkah 1: Strip spasi & lowercase dulu
df['Gender'] = df['Gender'].str.strip().str.lower()

# Langkah 2: Mapping alias ke bentuk standar
gender_map = {
    'male': 'Male',
    'm': 'Male',
    'female': 'Female',
    'f': 'Female',
    'non-binary': 'Non-binary',
    'prefer not to say': 'Prefer not to say'
}
df['Gender'] = df['Gender'].map(gender_map)

print('Gender setelah standarisasi:')
print(df['Gender'].value_counts(dropna=False))

Gender setelah standarisasi:
Gender
Male                 41713
Female               40809
NaN                   3161
Prefer not to say     2983
Non-binary            1318
Name: count, dtype: int64


In [24]:
print(df['Gender'].unique())

<StringArray>
['Female', 'Male', nan, 'Prefer not to say', 'Non-binary']
Length: 5, dtype: str


In [25]:
# 4B. CUSTOMER SEGMENT: 8 variasi -> 4 kategori
df['Customer_Segment'] = df['Customer_Segment'].str.strip().str.lower()
segment_map = {
    'returning': 'Returning',
    'new': 'New',
    'loyal': 'Loyal',
    'corporate': 'Corporate'
}
df['Customer_Segment'] = df['Customer_Segment'].map(segment_map)

print('Customer Segment setelah standarisasi:')
print(df['Customer_Segment'].value_counts(dropna=False))

Customer Segment setelah standarisasi:
Customer_Segment
Returning    37997
New          21434
Loyal        20496
Corporate     5936
NaN           4121
Name: count, dtype: int64


In [26]:
# 4C. SALES CHANNEL: 10 variasi -> 4 kategori
df['Sales_Channel'] = df['Sales_Channel'].str.strip().str.lower()
channel_map = {
    'online': 'Online',
    'e-commerce': 'Online',       # E-commerce = Online
    'marketplace': 'Marketplace',
    'retail store': 'Retail Store',
    'retail': 'Retail Store',      # Retail = Retail Store
    'phone order': 'Phone Order'
}
df['Sales_Channel'] = df['Sales_Channel'].map(channel_map)

print('Sales Channel setelah standarisasi:')
print(df['Sales_Channel'].value_counts(dropna=False))

Sales Channel setelah standarisasi:
Sales_Channel
Online          40727
Marketplace     21168
Retail Store    21125
Phone Order      5289
NaN              1675
Name: count, dtype: int64


In [27]:
# 4D. PAYMENT METHOD: 11 variasi -> 5 kategori
df['Payment_Method'] = df['Payment_Method'].str.strip().str.lower()
payment_map = {
    'cash on delivery': 'Cash on Delivery',
    'cod': 'Cash on Delivery',             # COD = Cash on Delivery
    'digital wallet': 'Digital Wallet',
    'wallet': 'Digital Wallet',             # Wallet = Digital Wallet
    'debit card': 'Debit Card',
    'credit card': 'Credit Card',
    'bank transfer': 'Bank Transfer'
}
df['Payment_Method'] = df['Payment_Method'].map(payment_map)

print('Payment Method setelah standarisasi:')
print(df['Payment_Method'].value_counts(dropna=False))

Payment Method setelah standarisasi:
Payment_Method
Cash on Delivery    28660
Digital Wallet      19539
Debit Card          16633
Credit Card         12490
Bank Transfer        6703
NaN                  5959
Name: count, dtype: int64


In [28]:
# 4E. MARKETING CHANNEL: 12 variasi -> 7 kategori
df['Marketing_Channel'] = df['Marketing_Channel'].str.strip().str.lower()
marketing_map = {
    'search': 'Search',
    'social media': 'Social Media',
    'social': 'Social Media',            # Social = Social Media
    'direct': 'Direct',
    'email': 'Email',
    'influencer': 'Influencer',
    'affiliate': 'Affiliate',
    'sms': 'SMS'
}
df['Marketing_Channel'] = df['Marketing_Channel'].map(marketing_map)

print('Marketing Channel setelah standarisasi:')
print(df['Marketing_Channel'].value_counts(dropna=False))

Marketing Channel setelah standarisasi:
Marketing_Channel
Search          19605
Social Media    17993
Direct          13228
Email            9831
Influencer       8306
Affiliate        7624
NaN              7619
SMS              5778
Name: count, dtype: int64


---
## STEP 5: Treatment Invalid Data
Data yang secara logika **tidak mungkin terjadi di dunia nyata**:
- Age > 120 tahun atau Age = 0
- Harga barang negatif
- Quantity negatif
- Diskon negatif atau > 100%

Treatment: Ubah ke `NaN` (jangan hapus baris, karena kolom lain di baris itu mungkin masih valid).

In [29]:
# 5A. Treatment Invalid Values -> ubah ke NaN
print('Sebelum treatment:')
print(f"  Age invalid   : {((df['Customer_Age'] == 0) | (df['Customer_Age'] > 120)).sum()}")
print(f"  Price negatif : {(df['Unit_Price'] < 0).sum()}")
print(f"  Qty <= 0      : {(df['Quantity'] <= 0).sum()}")
print(f"  Disc invalid  : {((df['Discount_Percent'] < 0) | (df['Discount_Percent'] > 100)).sum()}")

# Ubah nilai invalid menjadi NaN
df.loc[(df['Customer_Age'] == 0) | (df['Customer_Age'] > 120), 'Customer_Age'] = np.nan
df.loc[df['Unit_Price'] < 0, 'Unit_Price'] = np.nan
df.loc[df['Quantity'] <= 0, 'Quantity'] = np.nan
df.loc[(df['Discount_Percent'] < 0) | (df['Discount_Percent'] > 100), 'Discount_Percent'] = np.nan

print('\nSetelah treatment (semua invalid -> NaN):')
print(f"  Age invalid   : {((df['Customer_Age'] == 0) | (df['Customer_Age'] > 120)).sum()}")
print(f"  Price negatif : {(df['Unit_Price'] < 0).sum()}")
print(f"  Qty <= 0      : {(df['Quantity'] <= 0).sum()}")
print(f"  Disc invalid  : {((df['Discount_Percent'] < 0) | (df['Discount_Percent'] > 100)).sum()}")

Sebelum treatment:
  Age invalid   : 17
  Price negatif : 17
  Qty <= 0      : 35
  Disc invalid  : 15

Setelah treatment (semua invalid -> NaN):
  Age invalid   : 0
  Price negatif : 0
  Qty <= 0      : 0
  Disc invalid  : 0


---
## STEP 6: Deteksi Outlier

### Perbedaan Invalid Data vs Outlier:
- **Invalid:** Tidak mungkin terjadi (Age = 132, Price = -250). Pasti salah.
- **Outlier:** Mungkin saja terjadi tapi nilainya ekstrem (Unit Price = 233.600). Belum tentu salah!

### 2 Metode Deteksi:
1. **IQR (Interquartile Range):** Cocok untuk data yang tidak harus normal. Patokan: di luar Q1 - 1.5*IQR atau Q3 + 1.5*IQR.
2. **Z-Score:** Mengukur berapa standar deviasi jauhnya dari mean. Patokan: |Z| > 3.

In [30]:
# 6A. Deteksi Outlier dengan Metode IQR
def detect_outlier_iqr(dataframe, column):
    """Mendeteksi outlier menggunakan metode IQR dan menampilkan hasilnya."""
    data = dataframe[column].dropna()
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - (1.5 * IQR)
    upper = Q3 + (1.5 * IQR)
    outliers = data[(data < lower) | (data > upper)]
    
    print(f'--- {column} ---')
    print(f'  Q1          : {Q1:,.2f}')
    print(f'  Q3          : {Q3:,.2f}')
    print(f'  IQR         : {IQR:,.2f}')
    print(f'  Lower Bound : {lower:,.2f}')
    print(f'  Upper Bound : {upper:,.2f}')
    print(f'  Outlier     : {len(outliers)} baris ({(len(outliers)/len(data)*100):.2f}%)')
    print(f'  Range outlier: {outliers.min():,.2f} s/d {outliers.max():,.2f}')
    print()
    return lower, upper, outliers

# Jalankan untuk kolom-kolom numerik yang penting
target_cols = ['Unit_Price', 'Quantity', 'Shipping_Cost', 'Sales_Amount', 'Customer_Age']
print('========== DETEKSI OUTLIER METODE IQR ==========')
iqr_results = {}
for col in target_cols:
    lower, upper, outliers = detect_outlier_iqr(df, col)
    iqr_results[col] = {'lower': lower, 'upper': upper, 'count': len(outliers)}

========== DETEKSI OUTLIER METODE IQR ==========
--- Unit_Price ---
  Q1          : 2,140.18
  Q3          : 5,827.45
  IQR         : 3,687.28
  Lower Bound : -3,390.74
  Upper Bound : 11,358.38
  Outlier     : 4347 baris (4.97%)
  Range outlier: 11,664.91 s/d 233,600.87

--- Quantity ---
  Q1          : 1.00
  Q3          : 3.00
  IQR         : 2.00
  Lower Bound : -2.00
  Upper Bound : 6.00
  Outlier     : 743 baris (0.84%)
  Range outlier: 7.00 s/d 114.00

--- Shipping_Cost ---
  Q1          : 149.18
  Q3          : 292.70
  IQR         : 143.52
  Lower Bound : -66.10
  Upper Bound : 507.98
  Outlier     : 236 baris (0.28%)
  Range outlier: 508.61 s/d 764.57

--- Sales_Amount ---
  Q1          : 3,727.13
  Q3          : 11,827.67
  IQR         : 8,100.54
  Lower Bound : -8,423.67
  Upper Bound : 23,978.47
  Outlier     : 6217 baris (6.91%)
  Range outlier: 23,978.85 s/d 178,951.92

--- Customer_Age ---
  Q1          : 28.00
  Q3          : 44.00
  IQR         : 16.00
  Lower Bound :

In [31]:
# 6B. Deteksi Outlier dengan Metode Z-Score
def detect_outlier_zscore(dataframe, column, threshold=3):
    """Mendeteksi outlier menggunakan Z-Score."""
    data = dataframe[column].dropna()
    mean = data.mean()
    std = data.std()
    z_scores = ((data - mean) / std).abs()
    outliers = data[z_scores > threshold]
    
    print(f'--- {column} ---')
    print(f'  Mean     : {mean:,.2f}')
    print(f'  Std Dev  : {std:,.2f}')
    print(f'  Threshold: |Z| > {threshold}')
    print(f'  Outlier  : {len(outliers)} baris ({(len(outliers)/len(data)*100):.2f}%)')
    if len(outliers) > 0:
        print(f'  Range    : {outliers.min():,.2f} s/d {outliers.max():,.2f}')
    print()
    return outliers

print('========== DETEKSI OUTLIER METODE Z-SCORE (|Z| > 3) ==========')
for col in target_cols:
    detect_outlier_zscore(df, col)

========== DETEKSI OUTLIER METODE Z-SCORE (|Z| > 3) ==========
--- Unit_Price ---
  Mean     : 4,700.39
  Std Dev  : 4,586.03
  Threshold: |Z| > 3
  Outlier  : 2504 baris (2.86%)
  Range    : 18,459.40 s/d 233,600.87

--- Quantity ---
  Mean     : 2.39
  Std Dev  : 1.75
  Threshold: |Z| > 3
  Outlier  : 281 baris (0.32%)
  Range    : 8.00 s/d 114.00

--- Shipping_Cost ---
  Mean     : 221.18
  Std Dev  : 104.79
  Threshold: |Z| > 3
  Outlier  : 105 baris (0.13%)
  Range    : 535.86 s/d 764.57

--- Sales_Amount ---
  Mean     : 9,833.84
  Std Dev  : 10,889.43
  Threshold: |Z| > 3
  Outlier  : 1967 baris (2.19%)
  Range    : 42,507.91 s/d 178,951.92

--- Customer_Age ---
  Mean     : 36.33
  Std Dev  : 11.36
  Threshold: |Z| > 3
  Outlier  : 171 baris (0.20%)
  Range    : 71.00 s/d 105.00



In [32]:
# 6C. Lihat Detail Baris Outlier (Contoh: Unit_Price)
# Biar lu bisa investigasi apakah outlier ini data beneran atau error

Q1 = df['Unit_Price'].quantile(0.25)
Q3 = df['Unit_Price'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + (1.5 * IQR)

# Tampilkan baris dengan Unit_Price di atas upper bound
extreme_prices = df.loc[
    df['Unit_Price'] > upper_bound,
    ['Order_ID', 'Product_Name', 'Product_Category', 'Unit_Price', 'Quantity', 'Sales_Amount']
].sort_values('Unit_Price', ascending=False)

print(f'Upper Bound IQR: {upper_bound:,.2f}')
print(f'Jumlah outlier harga: {len(extreme_prices)}')
print('\n10 Harga Tertinggi (Apakah masuk akal?):')
display(extreme_prices.head(10))

Upper Bound IQR: 11,358.38
Jumlah outlier harga: 4347

10 Harga Tertinggi (Apakah masuk akal?):


,Order_ID,Product_Name,Product_Category,Unit_Price,Quantity,Sales_Amount
44023,ORD-0044024,Notebook Pack,Stationery,233600.87,2.0,1418.51
38798,ORD-0038799,Notebook Pack,Stationery,215968.87,4.0,2571.33
17515,ORD-0017516,Laptop Stand,Office Supplies,212789.76,4.0,12295.88
66989,ORD-0066990,Face Serum,Beauty,198702.15,1.0,2119.75
4855,ORD-0004856,NaN,Beauty,198353.49,7.0,16008.38
64607,ORD-0064608,NaN,Home & Kitchen,178835.90,2.0,8393.55
1592,ORD-0001593,Running Shoes,Clothing,164049.48,1.0,5647.16
73738,ORD-0073739,Building Blocks,Toys & Games,132238.12,3.0,7892.29
78539,ORD-0078540,Face Serum,Beauty,122389.35,2.0,4915.69
21919,ORD-0021920,Smart Watch Strap,Accessories,95864.95,1.0,1497.01


---
## STEP 7: Ringkasan Akhir & Export

### Penting: Outlier TIDAK otomatis dihapus!
```
Detect outlier
     |
Investigate (lihat datanya)
     |
Understand (apakah masuk akal secara bisnis?)
     |
Decide treatment (biarkan / cap / hapus / transformasi)
```
- Harga 233.600 untuk furniture premium? Mungkin **bukan** outlier.
- Quantity 114? Bisa jadi order bulk/grosir. Investigasi dulu.
- Profit margin negatif (-13.8%)? Bisa jadi memang rugi karena promo besar. **Valid.**

In [33]:
# Ringkasan kondisi data setelah cleaning
print('=== RINGKASAN DATA SETELAH CLEANING ===')
print(f'Ukuran dataset: {df.shape}')
print(f'\nMissing value tersisa:')
remaining = df.isnull().sum()
display(remaining[remaining > 0])

print(f'\nVariasi unik per kolom kategorikal (setelah standarisasi):')
for col in ['Gender', 'Customer_Segment', 'Sales_Channel', 'Payment_Method', 'Marketing_Channel']:
    vals = df[col].dropna().unique()
    print(f'  {col}: {len(vals)} -> {list(vals)}')

=== RINGKASAN DATA SETELAH CLEANING ===
Ukuran dataset: (89984, 23)

Missing value tersisa:


Customer_Age          4983
Gender                3161
City                  2205
Customer_Segment      4121
Sales_Channel         1675
Product_Category      1079
Product_Name          1672
Unit_Price            2533
Quantity              1033
Discount_Percent      4073
Shipping_Cost         6779
Payment_Method        5959
Order_Status          2020
Marketing_Channel     7619
Customer_Rating      22558
Profit_Margin         3224
dtype: int64


Variasi unik per kolom kategorikal (setelah standarisasi):
  Gender: 4 -> ['Female', 'Male', 'Prefer not to say', 'Non-binary']
  Customer_Segment: 4 -> ['Corporate', 'New', 'Returning', 'Loyal']
  Sales_Channel: 4 -> ['Retail Store', 'Marketplace', 'Online', 'Phone Order']
  Payment_Method: 5 -> ['Credit Card', 'Cash on Delivery', 'Debit Card', 'Digital Wallet', 'Bank Transfer']
  Marketing_Channel: 7 -> ['Direct', 'Social Media', 'Search', 'SMS', 'Email', 'Affiliate', 'Influencer']


In [34]:
# Export dataset hasil cleaning
output_path = r'D:\Data Analyst\Python\sales_cleaned.csv'
df.to_csv(output_path, index=False)
print('Dataset bersih disimpan ke:', output_path)
print('Dimensi akhir:', df.shape)

Dataset bersih disimpan ke: D:\Data Analyst\Python\sales_cleaned.csv
Dimensi akhir: (89984, 23)
